In [10]:
import pandas as pd
!python -m pip install scikit-surprise

import surprise
from surprise import Reader


In [11]:
book_ratings = pd.read_csv('goodreads_ratings.csv')
print(book_ratings.head())

print(len(book_ratings))
print(book_ratings.info())

print(book_ratings['rating'].value_counts())

book_ratings = book_ratings[(book_ratings['rating'] >= 1) & (book_ratings['rating'] <= 5)]


                            user_id   book_id  \
0  d089c9b670c0b0b339353aebbace46a1   7686667   
1  6dcb2c16e12a41ae0c6c38e9d46f3292  18073066   
2  244e0ce681148a7586d7746676093ce9  13610986   
3  73fcc25ff29f8b73b3a7578aec846394  27274343   
4  f8880e158a163388a990b64fec7df300  11614718   

                          review_id  rating  \
0  3337e0e75701f7f682de11638ccdc60c       3   
1  7201aa3c1161f2bad81258b6d4686c16       5   
2  07a203f87bfe1b65ff58774667f6f80d       5   
3  8be2d87b07098c16f9742020ec459383       1   
4  a29c4ba03e33ad073a414ac775266c5f       4   

                                         review_text  \
0  Like Matched, this book felt like it was echoi...   
1  WOW again! 4,5 Stars \r\n So i wont forget to ...   
2  The second novel was hot & heavy. Not only in ...   
3  What a maddening waste of time. And I unfortun...   
4  4.5 stars! \r\n This was an awesome read! \r\n...   

                       date_added                    date_updated  \
0  Fri Apr 29 14

In [12]:
#Prepare data for Surprise: build a Reader object
reader = Reader(rating_scale=(1, 5))

# Load `book_ratings` into a Surprise Dataset
from surprise import Dataset
rec_data = Dataset.load_from_df(book_ratings[['user_id',
                                              'book_id',
                                              'rating']],
                                reader)

#Create a 80:20 train-test split and set the random state to 7
from surprise.model_selection import train_test_split
trainset, testset = train_test_split(rec_data, test_size=.2, random_state=7)


In [13]:
#Use KNNBasic from Surprise to train a collaborative filter
from surprise import KNNBasic
nn_algo = KNNBasic(sim_options={'name': 'cosine', 'user_based': True}) 
nn_algo.fit(trainset)


Computing the cosine similarity matrix...
Done computing similarity matrix.


In [14]:
#Evaluate the recommender system
from surprise import accuracy
predictions = nn_algo.test(testset)
accuracy.rmse(predictions)


RMSE: 1.1105


np.float64(1.110471008157185)

In [15]:
#Prediction on a user who gave the "The Three-Body Problem" a rating of 5
pred = nn_algo.predict('8842281e1d1347389f2ab93d60773d4d', '18007564')
print(f"Predicted rating for 'The Martian': {pred.est}")


Predicted rating for 'The Martian': 3.8250739644970415


In [16]:
#Hyperparameter tuning loop to improve accuracy
for k in [20, 30, 40]:
    for sim in ['cosine', 'pearson']:
        algo = KNNBasic(k=k, sim_options={'name': sim, 'user_based': True})
        algo.fit(trainset)
        preds = algo.test(testset)
        rmse = accuracy.rmse(preds, verbose=False)
        print(f"k={k}, sim={sim}, RMSE={rmse:.4f}")


Computing the cosine similarity matrix...
Done computing similarity matrix.
k=20, sim=cosine, RMSE=1.1105
Computing the pearson similarity matrix...
Done computing similarity matrix.
k=20, sim=pearson, RMSE=1.1105
Computing the cosine similarity matrix...
Done computing similarity matrix.
k=30, sim=cosine, RMSE=1.1105
Computing the pearson similarity matrix...
Done computing similarity matrix.
k=30, sim=pearson, RMSE=1.1105
Computing the cosine similarity matrix...
Done computing similarity matrix.
k=40, sim=cosine, RMSE=1.1105
Computing the pearson similarity matrix...
Done computing similarity matrix.
k=40, sim=pearson, RMSE=1.1105
